# Module 02 — Exercises: Backpropagation
Each backward function takes the upstream gradient and returns gradients with the **same shapes** as the inputs.
The checks compare against numerical gradients.

In [ ]:
# Standard setup: make the repo root importable, pick a device, fix the random seed.
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parents[1]  # notebooks live in modules/XX_name/, so go up two levels
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from common.device import get_device, seed_everything, fast_dev_run

FAST_DEV_RUN = fast_dev_run()  # True only for automated checks; leave it False when learning
DATA_DIR = ROOT / "data"
device = get_device()
seed_everything(42)

In [ ]:
import numpy as np

rng = np.random.default_rng(2)


def num_grad(f, x, eps=1e-6):
    """Numerical gradient of scalar f() w.r.t. array x (modified in place, then restored)."""
    g = np.zeros_like(x)
    for idx in np.ndindex(x.shape):
        old = x[idx]
        x[idx] = old + eps; fp = f()
        x[idx] = old - eps; fm = f()
        x[idx] = old
        g[idx] = (fp - fm) / (2 * eps)
    return g

### Exercise 1 — Scalar graph by hand
For $f(a, b, c) = (a \cdot b + c)^2$ at a=2, b=-3, c=4, compute the forward value and all three partial
derivatives by hand with the chain rule (write the intermediate values as code).

In [ ]:
a, b, c = 2.0, -3.0, 4.0
f = None      # TODO
df_da = None  # TODO
df_db = None  # TODO
df_dc = None  # TODO

In [ ]:
assert f == 4.0
assert (df_da, df_db, df_dc) == (12.0, -8.0, -4.0)
print("Exercise 1 passed ✅")

### Exercise 2 — Linear layer backward
Forward: `Z = X @ W + b`. Given upstream `dZ`, return `(dX, dW, db)`.

In [ ]:
def linear_backward(dZ, X, W):
    # TODO
    raise NotImplementedError

In [ ]:
X = rng.normal(size=(5, 3)); W = rng.normal(size=(3, 4)); b = rng.normal(size=4)
R = rng.normal(size=(5, 4))                   # random projection -> scalar loss L = sum(Z * R), so dZ = R
L = lambda: np.sum((X @ W + b) * R)
dX, dW, db = linear_backward(R, X, W)
assert np.allclose(dX, num_grad(L, X), atol=1e-5)
assert np.allclose(dW, num_grad(L, W), atol=1e-5)
assert np.allclose(db, num_grad(L, b), atol=1e-5)
print("Exercise 2 passed ✅")

### Exercise 3 — Activation backward passes
Given the pre-activation `Z` and upstream `dA`, return `dZ` for ReLU, sigmoid and tanh.
Derivatives: σ'(z) = σ(z)(1−σ(z)), tanh'(z) = 1 − tanh²(z).

In [ ]:
def relu_backward(dA, Z):
    # TODO
    raise NotImplementedError


def sigmoid_backward(dA, Z):
    # TODO
    raise NotImplementedError


def tanh_backward(dA, Z):
    # TODO
    raise NotImplementedError

In [ ]:
Z = rng.normal(size=(4, 6)); R = rng.normal(size=(4, 6))
for fwd, bwd in [(lambda z: np.maximum(0, z), relu_backward),
                 (lambda z: 1 / (1 + np.exp(-z)), sigmoid_backward),
                 (np.tanh, tanh_backward)]:
    assert np.allclose(bwd(R, Z), num_grad(lambda: np.sum(fwd(Z) * R), Z), atol=1e-5), bwd.__name__
print("Exercise 3 passed ✅")

### Exercise 4 — Softmax + cross-entropy
Implement `softmax_ce(Z, y)` that returns `(loss, dZ)`, where loss is the **mean** cross-entropy over the batch and
`dZ` its gradient. Use the stable softmax.

In [ ]:
def softmax_ce(Z, y):
    # TODO
    raise NotImplementedError

In [ ]:
Z = rng.normal(size=(7, 4)); y = rng.integers(0, 4, size=7)
loss, dZ = softmax_ce(Z, y)
import torch
ref = torch.nn.functional.cross_entropy(torch.tensor(Z), torch.tensor(y))
assert np.isclose(loss, ref.item())
assert np.allclose(dZ, num_grad(lambda: softmax_ce(Z, y)[0], Z), atol=1e-6)
print("Exercise 4 passed ✅")

### Exercise 5 — A full 2-layer network (tanh hidden layer)
Network: `H = tanh(X @ W1 + b1)`, `Z = H @ W2 + b2`, loss = softmax-CE(Z, y).
Return the loss and a dict of gradients with keys `W1, b1, W2, b2`. Reuse your functions above.

In [ ]:
def two_layer_loss_and_grads(p, X, y):
    # TODO: forward (keep Z1 and H), then backward in reverse order
    raise NotImplementedError

In [ ]:
p = {"W1": rng.normal(size=(3, 5)), "b1": rng.normal(size=5), "W2": rng.normal(size=(5, 4)), "b2": rng.normal(size=4)}
X = rng.normal(size=(8, 3)); y = rng.integers(0, 4, size=8)
loss, g = two_layer_loss_and_grads(p, X, y)
for k in p:
    num = num_grad(lambda: two_layer_loss_and_grads(p, X, y)[0], p[k])
    assert g[k].shape == p[k].shape, k
    assert np.allclose(g[k], num, atol=1e-5), k
print("Exercise 5 passed ✅")

### Exercise 6 — Train it
Use your function to train on the XOR-like "checkerboard" data below with plain gradient descent.
Reach **≥ 95% training accuracy**.

In [ ]:
Xq = rng.uniform(-1, 1, size=(400, 2))
yq = ((Xq[:, 0] > 0) ^ (Xq[:, 1] > 0)).astype(int)   # XOR of the signs: label by quadrant

p = {"W1": rng.normal(0, 1, size=(2, 32)), "b1": np.zeros(32),
     "W2": rng.normal(0, 0.1, size=(32, 2)), "b2": np.zeros(2)}
# TODO: loop: compute loss & grads, update every p[k] -= lr * g[k]

In [ ]:
pred = (np.tanh(Xq @ p["W1"] + p["b1"]) @ p["W2"] + p["b2"]).argmax(axis=1)
acc = np.mean(pred == yq)
print(f"accuracy {acc:.1%}")
assert acc >= 0.95
print("Exercise 6 passed ✅")